# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [1]:
import math
import os
import random
import re
import time
import unicodedata
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 170)

STOPWORDS = set('''a about above after again against ain all am an and any are aren aren't as at be because been before being below between both but by can couldn couldn't d did didn didn't do does doesn doesn't doing don don't down during each few for from further had hadn hadn't has hasn hasn't have haven haven't having he he'd he'll he's her here hers herself him himself his how i i'd i'll i'm i've if in into is isn isn't it it'd it'll it's its itself just ll m ma me mightn mightn't more most must mustn't my myself needn needn't no nor not now o of off on once only or other our ours ourselves out over own re s same shan shan't she she'd she'll she's should should've shouldn shouldn't so some such t than that that'll the their theirs them themselves then there these they they'd they'll they're they've this those through to too under until up ve very was wasn wasn't we we'd we'll we're we've were weren weren't what when where which while who whom why will with won won't wouldn wouldn't y you you'd you'll you're you've your yours yourself yourselves'''.split())
TOKEN_RE = re.compile(r'[a-z]+')


def normalize_text(text):
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch)).lower()
    start = text.find('*** start of')
    end = text.find('*** end of')
    if start != -1:
        text = text[start:]
    if end != -1:
        text = text[:end]
    return text


def tokenize(text):
    return [
        token
        for token in TOKEN_RE.findall(normalize_text(text))
        if len(token) >= 3 and token not in STOPWORDS
    ]


LARGE_ROOT = Path('epub')
if not LARGE_ROOT.exists():
    LARGE_ROOT = Path('notebooks') / 'epub'
if not LARGE_ROOT.exists():
    raise FileNotFoundError(
        'No se encontró la carpeta epub. Ejecute el notebook desde la raíz del proyecto o desde notebooks/.'
    )

epub_paths = sorted(LARGE_ROOT.glob('*/*.txt'), key=lambda path: int(path.stem[2:]))
print(f'Archivos TXT encontrados: {len(epub_paths):,}')


def extract_metadata(path):
    title = author = language = None
    with open(path, 'r', encoding='utf-8', errors='ignore') as handle:
        for _ in range(200):
            line = handle.readline()
            if not line:
                break
            stripped = line.strip()
            if stripped.startswith('Title:') and title is None:
                title = stripped[6:].strip()
            elif stripped.startswith('Author:') and author is None:
                author = stripped[7:].strip()
            elif stripped.startswith('Language:') and language is None:
                language = stripped[9:].strip()
            if title and author:
                break
    return title, author, language


max_workers = min(32, (os.cpu_count() or 1) + 4)
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    epub_metadata = list(executor.map(extract_metadata, epub_paths))


def normalized_key(text):
    text = (text or '').lower()
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    return ' '.join(text.split())


def deduplication_key(path, title, author):
    key = (normalized_key(title), normalized_key(author))
    if not key[0] and not key[1]:
        key = ('__id__:' + path.parent.name, '')
    return key


records = []
seen_keys = {}
for path, (title, author, language) in zip(epub_paths, epub_metadata):
    doc_id = int(path.stem[2:])
    key = deduplication_key(path, title, author)
    is_duplicate = key in seen_keys
    if not is_duplicate:
        seen_keys[key] = path
    records.append(
        {
            'doc_id': doc_id,
            'path': path,
            'title': title,
            'author': author,
            'language': language,
            'bytes': path.stat().st_size,
            'duplicate': is_duplicate,
        }
    )

metadata_df = pd.DataFrame(records)
valid_df = metadata_df[~metadata_df['duplicate']].copy()
valid_df = valid_df.sort_values('doc_id').reset_index(drop=True)

SAMPLE_SIZE = 1000
rng = random.Random(42)
sample_positions = rng.sample(range(len(valid_df)), SAMPLE_SIZE)
working_df = valid_df.iloc[sample_positions].sort_values('doc_id').reset_index(drop=True)

doc_ids = [int(row.doc_id) for row in working_df.itertuples()]
paths = {int(row.doc_id): row.path for row in working_df.itertuples()}
titles = {
    int(row.doc_id): (
        row.title.strip()
        if isinstance(row.title, str) and row.title.strip()
        else f'pg{int(row.doc_id)}'
    )
    for row in working_df.itertuples()
}

start = time.perf_counter()
raw_texts = {}
tokenized_docs = {}
for doc_id in doc_ids:
    text = paths[doc_id].read_text(encoding='utf-8', errors='ignore')
    raw_texts[doc_id] = text
    tokenized_docs[doc_id] = tokenize(text)
load_seconds = time.perf_counter() - start

vocabulary = sorted({term for tokens in tokenized_docs.values() for term in tokens})
term_index = {term: index for index, term in enumerate(vocabulary)}

corpus_table = pd.DataFrame(
    {
        'Identificador de documento': doc_ids,
        'Título': [titles[doc_id] for doc_id in doc_ids],
        'Texto': [raw_texts[doc_id] for doc_id in doc_ids],
        'tokens': [tokenized_docs[doc_id] for doc_id in doc_ids],
    }
)

print(f'Documentos TXT totales: {len(metadata_df):,}')
print(f'Documentos válidos tras deduplicar: {len(valid_df):,}')
print(f'Muestra de trabajo: {len(doc_ids):,} documentos')
print(f'Tiempo de carga y tokenización: {load_seconds:,.2f} s')

preview = corpus_table.head(8).copy()
preview['Texto'] = preview['Texto'].str.replace(r'\s+', ' ', regex=True).str.slice(0, 90) + '…'
preview['tokens'] = preview['tokens'].apply(
    lambda tokens: f'[{len(tokens):,} tokens] ' + ' '.join(tokens[:8]) + ' …'
)
display(preview)
print('Vista previa: Texto y tokens truncados. La tabla completa conserva el texto y las listas de tokens.')

doc_lengths = np.array([len(tokenized_docs[doc_id]) for doc_id in doc_ids])
tf_counts = {doc_id: Counter(tokenized_docs[doc_id]) for doc_id in doc_ids}

df = {term: 0 for term in vocabulary}
for tokens in tokenized_docs.values():
    for term in set(tokens):
        df[term] += 1

N = len(doc_ids)
idf = {term: math.log(N / freq) + 1 for term, freq in df.items()}

print()
print(f'Número de documentos: {N:,}')
print(f'Tamaño del vocabulario: {len(vocabulary):,}')
print(f'Longitud mínima: {doc_lengths.min():,} términos')
print(f'Longitud media: {doc_lengths.mean():,.2f} términos')
print(f'Longitud máxima: {doc_lengths.max():,} términos')
print(f'Total de tokens procesados: {int(doc_lengths.sum()):,}')

sample_terms = ['one', 'love', 'whale', 'porpoise', 'cetacean']
display(
    pd.DataFrame(
        {
            'Término': sample_terms,
            'DF': [df[term] for term in sample_terms],
            'IDF': [round(idf[term], 4) for term in sample_terms],
        }
    )
)


Archivos TXT encontrados: 78,145


Documentos TXT totales: 78,145
Documentos válidos tras deduplicar: 77,148
Muestra de trabajo: 1,000 documentos
Tiempo de carga y tokenización: 70.47 s


,Identificador de documento,Título,Texto,tokens
0,59,Discourse on the Method of Rightly Conducting One's Reason and of Seeking Truth in the Sciences,The Project Gutenberg eBook of Discourse on the Method of Rightly Conducting One's Reason …,"[9,713 tokens] start project gutenberg ebook discourse method rightly conducting …"
1,81,The Return of Tarzan,The Project Gutenberg eBook of The Return of Tarzan This eBook is for the use of anyone an…,"[42,698 tokens] start project gutenberg ebook return tarzan return tarzan …"
2,163,Flower Fables,The Project Gutenberg eBook of Flower Fables This eBook is for the use of anyone anywhere …,"[18,090 tokens] start project gutenberg ebook flower fables illustration flower …"
3,257,Troilus and Criseyde,The Project Gutenberg eBook of Troilus and Criseyde This eBook is for the use of anyone an…,"[34,733 tokens] start project gutenberg ebook troilus criseyde produced douglas …"
4,318,John Barleycorn,The Project Gutenberg eBook of John Barleycorn This eBook is for the use of anyone anywher…,"[30,984 tokens] start project gutenberg ebook john barleycorn john barleycorn …"
5,372,"Prince Otto, a Romance","The Project Gutenberg eBook of Prince Otto, a Romance This eBook is for the use of anyone …","[29,260 tokens] start project gutenberg ebook prince otto romance transcribed …"
6,386,"Virginibus Puerisque, and Other Papers","The Project Gutenberg eBook of Virginibus Puerisque, and Other Papers This eBook is for th…","[23,120 tokens] start project gutenberg ebook virginibus puerisque papers transcribed …"
7,461,The Quest of the Golden Girl: A Romance,The Project Gutenberg eBook of The Quest of the Golden Girl: A Romance This eBook is for t…,"[24,490 tokens] start project gutenberg ebook quest golden girl romance …"


Vista previa: Texto y tokens truncados. La tabla completa conserva el texto y las listas de tokens.



Número de documentos: 1,000
Tamaño del vocabulario: 857,038
Longitud mínima: 73 términos
Longitud media: 29,975.46 términos
Longitud máxima: 391,045 términos
Total de tokens procesados: 29,975,463


,Término,DF,IDF
0,one,1000,1.0000
1,love,677,1.3901
2,whale,107,3.2349
3,porpoise,29,4.5405
4,cetacean,4,6.5215


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [2]:
import math

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse

pd.set_option('display.max_colwidth', 80)

N = len(doc_ids)
avgdl = float(doc_lengths.mean())

tfidf_rows, tfidf_cols, tfidf_values = [], [], []
tf_rows, tf_cols, tf_values = [], [], []
for row, doc_id in enumerate(doc_ids):
    for term, count in tf_counts[doc_id].items():
        column = term_index[term]
        tf_rows.append(row)
        tf_cols.append(column)
        tf_values.append(count)
        tfidf_rows.append(row)
        tfidf_cols.append(column)
        tfidf_values.append(count * idf[term])

tfidf_matrix = sparse.csr_matrix(
    (tfidf_values, (tfidf_rows, tfidf_cols)),
    shape=(N, len(vocabulary)),
)
tf_matrix = sparse.csc_matrix(
    (tf_values, (tf_rows, tf_cols)),
    shape=(N, len(vocabulary)),
)

doc_norms = np.sqrt(np.asarray(tfidf_matrix.multiply(tfidf_matrix).sum(axis=1)).ravel())
doc_norms[doc_norms == 0] = 1.0

K1 = 1.2
B = 0.75

idf_bm25 = {
    term: math.log(1 + (N - freq + 0.5) / (freq + 0.5))
    for term, freq in df.items()
}

print(
    f'Matriz TF-IDF dispersa: {tfidf_matrix.shape[0]:,} × {tfidf_matrix.shape[1]:,} '
    f'({tfidf_matrix.nnz:,} valores no nulos)'
)
print(f'Longitud media del corpus (avgdl): {avgdl:,.2f} términos')
print(f'Parámetros BM25: k1 = {K1}, b = {B}')


def df_category(freq):
    if freq == 0:
        return 'fuera del vocabulario'
    if freq >= 600:
        return 'muy frecuente'
    if freq >= 100:
        return 'frecuente'
    return 'poco frecuente'


QUERIES = [
    {
        'consulta': 'love',
        'intencion': 'Novelas y reflexiones sobre el amor y los afectos.',
    },
    {
        'consulta': 'porpoise',
        'intencion': 'Historia natural de las marsopas y pequeños cetáceos.',
    },
    {
        'consulta': 'whale sea',
        'intencion': 'Relatos de caza de ballenas y navegación marina.',
    },
    {
        'consulta': 'time life',
        'intencion': 'Obras sobre el paso del tiempo y el sentido de la vida.',
    },
    {
        'consulta': 'king queen death',
        'intencion': 'Realeza, poder monárquico y muerte.',
    },
    {
        'consulta': 'cetacean porpoise iceberg',
        'intencion': 'Cetáceos y fauna de los mares polares.',
    },
    {
        'consulta': 'dinosaur mammoth submarine microscope',
        'intencion': 'Fauna prehistórica y ciencia natural.',
    },
    {
        'consulta': 'smartphone wifi',
        'intencion': 'Tecnología moderna que no aparece en el corpus.',
    },
]


def build_query_vector(query):
    counts = Counter(tokenize(query))
    vector = np.zeros(len(vocabulary), dtype=float)
    for term, count in counts.items():
        index = term_index.get(term)
        if index is not None:
            vector[index] = count * idf[term]
    return counts, vector


def cosine_scores(vector):
    dot_products = np.asarray(tfidf_matrix @ vector).ravel()
    query_norm = float(np.linalg.norm(vector))
    denominator = query_norm * doc_norms
    return np.divide(
        dot_products,
        denominator,
        out=np.zeros_like(dot_products),
        where=denominator != 0,
    )


def bm25_scores(query_terms, k1=K1, b=B):
    scores = np.zeros(N, dtype=float)
    for term in query_terms:
        index = term_index.get(term)
        if index is None:
            continue
        frequency = tf_matrix[:, index].toarray().ravel().astype(float)
        denominator = frequency + k1 * (1 - b + b * doc_lengths / avgdl)
        scores += idf_bm25[term] * (frequency * (k1 + 1)) / np.where(
            denominator == 0, 1.0, denominator
        )
    return scores


def rank_positive(scores, limit=10):
    order = sorted(
        range(N),
        key=lambda position: (-scores[position], str(doc_ids[position])),
    )
    return [
        (doc_ids[position], float(scores[position]))
        for position in order
        if scores[position] > 0
    ][:limit]


summary_rows = []
rankings = {}
for entry in QUERIES:
    query = entry['consulta']
    tokens = tokenize(query)
    _, query_vector = build_query_vector(query)
    cos_scores = cosine_scores(query_vector)
    bm_scores = bm25_scores(tokens)
    tfidf_hits = rank_positive(cos_scores)
    bm25_hits = rank_positive(bm_scores)
    in_vocabulary = [term for term in tokens if term in term_index]
    term_df = '; '.join(
        f'{term}={df.get(term, 0)} ({df_category(df.get(term, 0))})'
        for term in tokens
    )
    rankings[query] = {
        'intencion': entry['intencion'],
        'tokens': tokens,
        'coseno': tfidf_hits,
        'bm25': bm25_hits,
        'cos_scores': cos_scores,
        'bm25_scores': bm_scores,
    }
    summary_rows.append(
        {
            'Consulta': query,
            'Qué busca el usuario': entry['intencion'],
            'Longitud': len(tokens),
            'Términos y DF': term_df,
            'En vocabulario': f'{len(in_vocabulary)}/{len(tokens)}',
            'Resultados TF-IDF': len(tfidf_hits),
            'Resultados BM25': len(bm25_hits),
        }
    )

display(pd.DataFrame(summary_rows))

tfidf_columns = ['Posición', 'Identificador', 'Título', 'Coseno']
bm25_columns = ['Posición', 'Identificador', 'Título', 'BM25']

for query, data in rankings.items():
    print()
    print(f'Consulta: {query!r} — {data["intencion"]}')
    print('TF-IDF + coseno:')
    if data['coseno']:
        display(
            pd.DataFrame(
                [
                    {
                        'Posición': position,
                        'Identificador': doc_id,
                        'Título': titles[doc_id],
                        'Coseno': round(score, 6),
                    }
                    for position, (doc_id, score) in enumerate(data['coseno'], start=1)
                ]
            )
        )
    else:
        display(pd.DataFrame(columns=tfidf_columns))
    print('BM25:')
    if data['bm25']:
        display(
            pd.DataFrame(
                [
                    {
                        'Posición': position,
                        'Identificador': doc_id,
                        'Título': titles[doc_id],
                        'BM25': round(score, 6),
                    }
                    for position, (doc_id, score) in enumerate(data['bm25'], start=1)
                ]
            )
        )
    else:
        display(pd.DataFrame(columns=bm25_columns))


Matriz TF-IDF dispersa: 1,000 × 857,038 (6,069,674 valores no nulos)
Longitud media del corpus (avgdl): 29,975.46 términos
Parámetros BM25: k1 = 1.2, b = 0.75


,Consulta,Qué busca el usuario,Longitud,Términos y DF,En vocabulario,Resultados TF-IDF,Resultados BM25
0,love,Novelas y reflexiones sobre el amor y los afectos.,1,love=677 (muy frecuente),1/1,10,10
1,porpoise,Historia natural de las marsopas y pequeños cetáceos.,1,porpoise=29 (poco frecuente),1/1,10,10
2,whale sea,Relatos de caza de ballenas y navegación marina.,2,whale=107 (frecuente); sea=650 (muy frecuente),2/2,10,10
3,time life,Obras sobre el paso del tiempo y el sentido de la vida.,2,time=786 (muy frecuente); life=771 (muy frecuente),2/2,10,10
4,king queen death,"Realeza, poder monárquico y muerte.",3,king=588 (frecuente); queen=467 (frecuente); death=687 (muy frecuente),3/3,10,10
5,cetacean porpoise iceberg,Cetáceos y fauna de los mares polares.,3,cetacean=4 (poco frecuente); porpoise=29 (poco frecuente); iceberg=28 (poco ...,3/3,10,10
6,dinosaur mammoth submarine microscope,Fauna prehistórica y ciencia natural.,4,dinosaur=4 (poco frecuente); mammoth=50 (poco frecuente); submarine=55 (poco...,4/4,10,10
7,smartphone wifi,Tecnología moderna que no aparece en el corpus.,2,smartphone=0 (fuera del vocabulario); wifi=0 (fuera del vocabulario),0/2,0,0



Consulta: 'love' — Novelas y reflexiones sobre el amor y los afectos.
TF-IDF + coseno:


,Posición,Identificador,Título,Coseno
0,1,20861,Friendship,0.271679
1,2,44891,"The Works of the Right Honourable John, Earl of Rochester",0.247186
2,3,33943,Woman,0.247134
3,4,36915,His Lady of the Sonnets,0.231556
4,5,27349,Personal Friendships of Jesus,0.219592
5,6,39844,The Ascent of Man,0.210726
6,7,48893,"Mademoiselle de Maupin, Volume 1 (of 2)",0.165467
7,8,62084,"The Deceased Wife's Sister, and My Beautiful Neighbour, v. 3",0.161563
8,9,52865,Home Poems,0.160283
9,10,53435,"At Minas Basin, and Other Poems",0.147972


BM25:


,Posición,Identificador,Título,BM25
0,1,27349,Personal Friendships of Jesus,0.856047
1,2,20861,Friendship,0.855555
2,3,257,Troilus and Criseyde,0.854855
3,4,33943,Woman,0.854440
4,5,26991,A Short System of English Grammar,0.853748
5,6,77224,A noble sacrifice,0.853531
6,7,44891,"The Works of the Right Honourable John, Earl of Rochester",0.853503
7,8,10812,The Worshipper of the Image,0.853328
8,9,48893,"Mademoiselle de Maupin, Volume 1 (of 2)",0.853200
9,10,62084,"The Deceased Wife's Sister, and My Beautiful Neighbour, v. 3",0.853076


... (24 outputs left out)


Consulta: 'smartphone wifi' — Tecnología moderna que no aparece en el corpus.
TF-IDF + coseno:


,Posición,Identificador,Título,Coseno


BM25:


,Posición,Identificador,Título,BM25


**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.